# Peak memory and runtime of the torchpalmm SPT model

The bleeding-on-probing model of `fit.ipynb` (10,946 visits, 883 patients; six class $\times$ sex follow-up curves with a knot at every distinct follow-up value on the 0.1-year grid; fixed effects; random intercept plus random slopes on age at visit and log time since the preceding visit with unstructured covariance; iid residuals) is fitted in its own process under `/usr/bin/time -l` (macOS: peak resident set size). The process reads the prepared sample and builds its own model matrices, so matrix construction is part of the cost; a "baseline" run (start-up, imports and data only) is subtracted. The sample is prepared once below, outside the measured process.


In [1]:
## settings, analysis sample (same rows and covariates as fit.ipynb) and helpers
import subprocess, re, sys, os
import numpy as np, pandas as pd
os.makedirs("benchmark/out", exist_ok=True)
data_csv = os.path.abspath("benchmark/spt_analysis_sample.csv")

d = pd.read_csv("spt_data/curated_spt_bleeding.csv")
d = d[d["spt_id"] >= 2].dropna(subset=["bleeding_pct"]).sort_values(["pat_id", "spt_id"]).reset_index(drop=True)   # first visit excluded, missing outcome dropped; age 90 kept, no interval filter
s = pd.DataFrame({"pat_id": d["pat_id"], "spt_id": d["spt_id"], "y": np.log1p(d["bleeding_pct"]),
                  "fu": ((d["cum_followup_years"] / 0.1).round() * 0.1).round(10)})            # follow-up time on the 0.1-year grid
cont = {"age_z": d["age_at_spt"], "year_z": d["calendar_year"], "teeth_z": d["n_teeth_spt"], "logdays_z": np.log(d["days_since_prev_visit"])}
for k, v in cont.items():
    s[k] = (v - v.mean()) / v.std()                                  # standardized as in the notebook
cls = d["baseline_diagnosis"].astype(int).astype(str); fem = (d["sex"] == "female").astype(float)
s["cls2"], s["cls3"], s["female"] = (cls == "2").astype(float), (cls == "3").astype(float), fem
s["ever"] = (d["smoking_status"] != "non-smoker").astype(float)
s["cls2_female"], s["cls3_female"] = s["cls2"] * fem, s["cls3"] * fem
s["grp"] = "class " + cls + ", " + d["sex"]
assert s.notna().all().all()
s.to_csv(data_csv, index=False)
print(f"{len(s)} visits, {s['pat_id'].nunique()} patients; distinct follow-up values per group:", s.groupby("grp")["fu"].nunique().to_dict())

def run(cmd):
    """Run cmd under /usr/bin/time -l; return wall seconds, peak memory (GB) and the RESULT line."""
    res = subprocess.run(["/usr/bin/time", "-l"] + cmd, capture_output=True, text=True)
    peak, wall = re.search(r"(\d+)\s+maximum resident set size", res.stderr), re.search(r"([\d.]+)\s+real", res.stderr)
    if res.returncode or not peak:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{res.stderr[-1500:]}")
    result = re.search(r"RESULT .*", res.stdout)
    return {"wall (s)": float(wall.group(1)), "peak memory (GB)": int(peak.group(1)) / 1e9, "result": result.group(0) if result else ""}

10946 visits, 883 patients; distinct follow-up values per group: {'class 1, female': 180, 'class 1, male': 157, 'class 2, female': 225, 'class 2, male': 243, 'class 3, female': 232, 'class 3, male': 220}


In [2]:
## run: baseline and fit, each in its own process
py = [sys.executable, "benchmark/benchmark_spt_torchpalmm.py"]
runs = {"baseline": py + ["baseline", data_csv, "benchmark/out/spt_tp.csv"],
        "fit": py + ["fit", data_csv, "benchmark/out/spt_tp.csv"]}                  # matrices + REML fit
measured = {key: run(cmd) for key, cmd in runs.items()}
base = measured["baseline"]["peak memory (GB)"]
tab = pd.DataFrame([{"run": r, "wall (s)": v["wall (s)"], "peak memory (GB)": v["peak memory (GB)"], "peak minus baseline (GB)": v["peak memory (GB)"] - base}
                    for r, v in measured.items()]).set_index("run")
print(tab.round(3).to_string())
for key, r in measured.items():
    print(key, "|", r["result"])

          wall (s)  peak memory (GB)  peak minus baseline (GB)
run                                                           
baseline      0.75             0.263                     0.000
fit          45.67             3.685                     3.421
baseline | RESULT mode=baseline rows=10946
fit | RESULT mode=fit rows=10946 subjects=883 total_basis=1245 woodbury=True epochs=25 lbfgs_steps=224 func_evals=281 fit_seconds=44.8 summary_seconds=0.0
